# Testing whether `rechp.maxeval = 2000` is actually needed

`rechp.xtol_rel = rechp.ftol_rel = rechp.ftol_abs = 0.0` in `scripts/endtoend_opt.jl`'s production
config, so the inner reconstruction (`reconstruct_Tmap_and_zmap_distributed`) currently never stops
early for any reason other than hitting `maxeval` -- confirmed by `MAXEVAL_REACHED` showing up on
nearly every outer iteration of the real run (`optdata/opt_2026-11-13T13:07:16.666/history.csv`),
which is a large part of why each outer iteration costs ~30 minutes.

**The heavy computation lives in `scripts/test_reconstruction_xtol.jl`, run via
`sbatch cluster_scripts/run_test_reconstruction_xtol.sh`, not here** -- same reasoning as the
`PSF_zlen`/`freq_order` convergence studies: it needs real distributed workers
(`reconstruct_Tmap_and_zmap_distributed`'s dominant cost, `make_image_from_3D_distributed`, `pmap`s
across `workers()` on every inner NLopt iteration), which the standing interactive Jupyter
session's own SLURM allocation is too small to provide (`NumNodes=1, NumCPUs=2, NumTasks=1` --
`SlurmManager` has no spare task slot to launch a worker into). This notebook just loads that
script's saved results and plots.

That script sweeps `xtol_rel ∈ {1e-3,...,1e-7}` (`ftol_rel`/`ftol_abs` left at `0.0` -- see the
planning discussion for why `xtol_rel` specifically, not those: they have a documented history of
causing spurious premature `:SUCCESS` in this exact pipeline, see the comment above
`reconstruct_Tmap_and_zmap_distributed`, `src/forward_distributed.jl`) against both the run's
"initial" geoms/α/β (pre-training) and its "checkpoint" geoms/α/β (iteration 150, the last
checkpoint before the job hit SLURM's wall-clock limit), using one shared random test object/noise
draw reused across every comparison. `maxeval` is kept at 2000 throughout as a safety ceiling, not
the thing driving convergence.

In [ ]:
cd(isdir("../../End2EndThermalImg.jl") ? "../../End2EndThermalImg.jl" : "../../End2EndThermalImg")
import Pkg
Pkg.activate(".")

In [ ]:
using End2EndThermalImg
using JLD2
using PythonPlot
using DataFrames

## Load precomputed results

Picks the most recent `convergence_studies/xtol_rel_<datetime>/` run folder by default -- set
`run_folder` explicitly instead if you want a specific earlier run.

In [ ]:
root = "convergence_studies"
run_names = sort(filter(name -> startswith(name, "xtol_rel_"), readdir(root)))
run_folder = joinpath(root, run_names[end])
println("Loading results from: ", run_folder)

results_file = joinpath(run_folder, "results.jld2")
source_run_folder, checkpoint_iteration, xtol_rels, maxeval_ceiling, object, noise, summary_rows,
    object_opts, image_opts, noisy_images, php, imghp =
    JLD2.load(results_file, "source_run_folder", "checkpoint_iteration", "xtol_rels", "maxeval_ceiling",
              "object", "noise", "summary_rows", "object_opts", "image_opts", "noisy_images", "php", "imghp")

println("source_run_folder = ", source_run_folder, "  (checkpoint iteration ", checkpoint_iteration, ")")
println("xtol_rels = ", xtol_rels, "  maxeval_ceiling = ", maxeval_ceiling)

df = DataFrame(summary_rows)
state_names = unique(df.state)

## Summary table

In [ ]:
df

## Reconstruction error and iteration count vs. `xtol_rel`

The main answer to "do we need `maxeval = 2000`": does `L` stay flat down to some looser
tolerance, and at what iteration count does that tolerance actually stop? Any point sitting on the
`maxeval` ceiling line (right panel) means that `xtol_rel` value was too tight to matter at this
budget.

In [ ]:
fig, axes = subplots(1, 2; figsize=(11, 4.5))

for state_name in state_names
    sub = df[df.state .== state_name, :]
    axes[0].plot(sub.xtol_rel, sub.L, marker="o", label=state_name)
    axes[1].plot(sub.xtol_rel, sub.n_iterations, marker="o", label=state_name)
end

axes[0].set_xscale("log")
axes[0].set_xlabel("xtol_rel")
axes[0].set_ylabel("L(object, object_opt)")
axes[0].set_title("Reconstruction error vs. xtol_rel")
axes[0].legend()
axes[0].grid(true, alpha=0.3)

axes[1].set_xscale("log")
axes[1].set_xlabel("xtol_rel")
axes[1].set_ylabel("Iterations used")
axes[1].axhline(maxeval_ceiling, color="gray", linestyle="--", label="maxeval ceiling")
axes[1].set_title("Iteration count vs. xtol_rel")
axes[1].legend()
axes[1].grid(true, alpha=0.3)

fig.tight_layout()
display(fig)

## Visual comparison: loosest vs. tightest tolerance

Reconstructed T/z maps and the formed-image/data-misfit row, for the loosest and tightest
`xtol_rel` tested, for each geoms state -- all from precomputed `object_opts`/`image_opts`, no
distributed computation needed here.

In [ ]:
tol_loose, tol_tight = maximum(xtol_rels), minimum(xtol_rels)

for state_name in state_names
    noisy_image = noisy_images[state_name]
    for tol in (tol_loose, tol_tight)
        object_opt = object_opts[(state_name, tol)]
        image_opt = image_opts[(state_name, tol)]
        fig, ax = End2EndThermalImg.plot_reconstruction_results(object, object_opt, noisy_image, image_opt;
            extra_title = "$state_name | xtol_rel = $tol")
        display(fig)
    end
end